In [ ]:
from pathlib import Path
import zipfile
import shutil
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

RAW_DIR = DATA_DIR / "aportacion_jaime_raw"
EXTRACTED_DIR = RAW_DIR / "extracted"

REVISION_DIR = OUTPUT_DIR / "revision_mario"
TABLES_DIR = OUTPUT_DIR / "tables"

FINAL_DIR = DATA_DIR / "imgsMIIA_jaime"
FINAL_IMAGES_DIR = FINAL_DIR / "images"

ZIP_PATH = RAW_DIR / "jimenez_mario.zip"

for carpeta in [RAW_DIR, EXTRACTED_DIR, REVISION_DIR, TABLES_DIR, FINAL_IMAGES_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("ZIP_PATH existe:", ZIP_PATH.exists())
print("ZIP_PATH:", ZIP_PATH)

In [ ]:
if EXTRACTED_DIR.exists():
    shutil.rmtree(EXTRACTED_DIR)

EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACTED_DIR)

print("Zip descomprimido en:")
print(EXTRACTED_DIR)

In [ ]:
ROOT_MARIO = EXTRACTED_DIR / "jimenez_mario"

imagenes = sorted(ROOT_MARIO.glob("mario_jimenez_*.jpg"))
mascaras = sorted(ROOT_MARIO.glob("mario_jimenez_*_mask.png"))

print("Imágenes:", len(imagenes))
print("Máscaras:", len(mascaras))

print(imagenes[:3])
print(mascaras[:3])

In [ ]:
filas = []

for img_path in imagenes:
    stem = img_path.stem
    mask_path = ROOT_MARIO / f"{stem}_mask.png"
    
    if not mask_path.exists():
        continue
    
    filas.append({
        "id": len(filas) + 1,
        "image_name": img_path.name,
        "mask_name": mask_path.name,
        "clase_lengua": "REVISAR",
        "usar": "SI",
        "comentario": ""
    })

df_revision = pd.DataFrame(filas)

csv_revision = TABLES_DIR / "mario_revision_clases.csv"
df_revision.to_csv(csv_revision, index=False)

print("CSV de revisión guardado en:")
print(csv_revision)

display(df_revision.head())

In [ ]:
def extraer_mascara_lengua(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    if mask is None:
        return None
    
    tongue = (mask == 2).astype(np.uint8) * 255
    
    return tongue

In [ ]:
def crear_overlay_lengua(img_path, mask_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    tongue = extraer_mascara_lengua(mask_path)
    
    if img is None or tongue is None:
        return None
    
    overlay = img.copy()
    color = np.zeros_like(img)
    color[:, :, 2] = 255
    color[:, :, 1] = 120
    
    mask_bool = tongue > 0
    overlay[mask_bool] = cv2.addWeighted(img, 0.45, color, 0.55, 0)[mask_bool]
    
    return overlay

In [ ]:
df_revision = pd.read_csv(csv_revision)

por_hoja = 20
num_hojas = int(np.ceil(len(df_revision) / por_hoja))

for h in range(num_hojas):
    df_hoja = df_revision.iloc[h * por_hoja:(h + 1) * por_hoja]
    
    plt.figure(figsize=(15, 12))
    
    for j, fila in enumerate(df_hoja.itertuples()):
        img_path = ROOT_MARIO / fila.image_name
        mask_path = ROOT_MARIO / fila.mask_name
        
        overlay = crear_overlay_lengua(img_path, mask_path)
        
        if overlay is None:
            continue
        
        overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
        
        plt.subplot(4, 5, j + 1)
        plt.imshow(overlay_rgb)
        plt.title(f"id={fila.id}", fontsize=9)
        plt.axis("off")
    
    plt.tight_layout()
    
    out_path = REVISION_DIR / f"mario_revision_sheet_{h+1:02d}.jpg"
    plt.savefig(out_path, dpi=150)
    plt.close()
    
    print("Guardada:", out_path)

In [ ]:
df_revision = pd.read_csv(csv_revision)

print("Clases:")
print(df_revision["clase_lengua"].value_counts(dropna=False))

print("\nUsar:")
print(df_revision["usar"].value_counts(dropna=False))

display(df_revision.head())

In [ ]:
clases_validas = ["DT", "VT", "LT", "DESCARTAR"]

pendientes = df_revision[~df_revision["clase_lengua"].isin(clases_validas)]

print("Pendientes o clases mal escritas:", len(pendientes))

if len(pendientes) > 0:
    display(pendientes)
    raise ValueError("Todavía hay imágenes sin clasificar. Rellena el CSV antes de seguir.")

print("Todo clasificado correctamente.")

In [ ]:
def mascara_a_regions_via(mask, area_min=80):
    regions = []
    
    if mask is None:
        return regions
    
    mask_bin = (mask > 0).astype(np.uint8) * 255
    
    contornos, _ = cv2.findContours(
        mask_bin,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    
    contornos = sorted(contornos, key=cv2.contourArea, reverse=True)
    
    for contorno in contornos:
        area = cv2.contourArea(contorno)
        
        if area < area_min:
            continue
        
        epsilon = 0.002 * cv2.arcLength(contorno, True)
        contorno_aprox = cv2.approxPolyDP(contorno, epsilon, True)
        
        puntos = contorno_aprox.reshape(-1, 2)
        
        if len(puntos) < 3:
            continue
        
        xs = puntos[:, 0].astype(int).tolist()
        ys = puntos[:, 1].astype(int).tolist()
        
        region = {
            "shape_attributes": {
                "name": "polygon",
                "all_points_x": xs,
                "all_points_y": ys
            },
            "region_attributes": {}
        }
        
        regions.append(region)
    
    return regions

In [ ]:
def crear_json_via_vacio():
    data = {
        "_via_settings": {
            "ui": {},
            "core": {},
            "project": {
                "name": "imgsMIIA_jaime"
            }
        },
        "_via_img_metadata": {},
        "_via_attributes": {
            "region": {},
            "file": {}
        },
        "_via_data_format_version": "2.0.10",
        "_via_image_id_list": []
    }
    
    return data

In [ ]:
if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)

FINAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta final limpia:")
print(FINAL_DIR)

In [ ]:
jsons = {
    "DT": crear_json_via_vacio(),
    "VT": crear_json_via_vacio(),
    "LT": crear_json_via_vacio()
}

filas_finales = []
filas_descartadas = []
contador = 1

for fila in df_revision.itertuples():
    clase = str(fila.clase_lengua).strip().upper()
    usar = str(fila.usar).strip().upper()
    
    img_path = ROOT_MARIO / fila.image_name
    mask_path = ROOT_MARIO / fila.mask_name
    
    if usar != "SI" or clase == "DESCARTAR":
        filas_descartadas.append({
            "id": fila.id,
            "image_name": fila.image_name,
            "mask_name": fila.mask_name,
            "clase_lengua": clase,
            "comentario": fila.comentario
        })
        continue
    
    if clase not in ["DT", "VT", "LT"]:
        filas_descartadas.append({
            "id": fila.id,
            "image_name": fila.image_name,
            "mask_name": fila.mask_name,
            "clase_lengua": clase,
            "comentario": "clase_no_valida"
        })
        continue
    
    tongue = extraer_mascara_lengua(mask_path)
    regions = mascara_a_regions_via(tongue)
    
    if len(regions) == 0:
        filas_descartadas.append({
            "id": fila.id,
            "image_name": fila.image_name,
            "mask_name": fila.mask_name,
            "clase_lengua": clase,
            "comentario": "sin_region_lengua"
        })
        continue
    
    nuevo_nombre = f"jaime_{contador:04d}_{clase}.jpg"
    img_out = FINAL_IMAGES_DIR / nuevo_nombre
    
    shutil.copy2(img_path, img_out)
    
    size = img_out.stat().st_size
    key = nuevo_nombre + str(size)
    
    jsons[clase]["_via_img_metadata"][key] = {
        "filename": nuevo_nombre,
        "size": size,
        "regions": regions,
        "file_attributes": {}
    }
    
    jsons[clase]["_via_image_id_list"].append(key)
    
    filas_finales.append({
        "id_original": fila.id,
        "image_original": fila.image_name,
        "mask_original": fila.mask_name,
        "image_final": nuevo_nombre,
        "clase_lengua": clase,
        "n_regions": len(regions)
    })
    
    contador += 1

df_final = pd.DataFrame(filas_finales)
df_descartadas = pd.DataFrame(filas_descartadas)

print("Imágenes finales:", len(df_final))
print("Descartadas:", len(df_descartadas))

print("\nDistribución final:")
print(df_final["clase_lengua"].value_counts())

display(df_final.head())

In [ ]:
for clase in ["DT", "VT", "LT"]:
    json_path = FINAL_DIR / f"{clase}.json"
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(jsons[clase], f, ensure_ascii=False)
    
    print("Guardado:", json_path)

In [ ]:
csv_final = FINAL_DIR / "resumen_imgsMIIA_jaime.csv"
csv_descartadas = FINAL_DIR / "descartadas.csv"

df_final.to_csv(csv_final, index=False)
df_descartadas.to_csv(csv_descartadas, index=False)

readme_path = FINAL_DIR / "README.txt"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write("imgsMIIA_jaime\n")
    f.write("================\n\n")
    f.write("Dataset adaptado a partir de jimenez_mario.zip.\n")
    f.write("El dataset original contenía máscaras con varias clases: fondo, dientes y lengua.\n")
    f.write("En esta versión se ha extraído únicamente la clase lengua y se ha reclasificado manualmente en DT, VT y LT.\n")
    f.write("Las imágenes dudosas o no viables se han dejado fuera y aparecen registradas en descartadas.csv.\n\n")
    f.write("Estructura:\n")
    f.write("- images/: imágenes finales usadas.\n")
    f.write("- DT.json: anotaciones VIA de lengua dorsal.\n")
    f.write("- VT.json: anotaciones VIA de lengua ventral.\n")
    f.write("- LT.json: anotaciones VIA de lengua lateral.\n")
    f.write("- descartadas.csv: imágenes no utilizadas.\n")

print("CSV final:", csv_final)
print("CSV descartadas:", csv_descartadas)
print("README:", readme_path)

In [ ]:
print("Imágenes en carpeta final:", len(list(FINAL_IMAGES_DIR.glob("*.jpg"))))

for clase in ["DT", "VT", "LT"]:
    json_path = FINAL_DIR / f"{clase}.json"
    
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    n = len(data["_via_img_metadata"])
    print(clase, ":", n)

In [ ]:
muestras = df_final.sample(
    n=min(8, len(df_final)),
    random_state=7
).reset_index(drop=True)

plt.figure(figsize=(12, 8))

for i, fila in muestras.iterrows():
    img_path = FINAL_IMAGES_DIR / fila["image_final"]
    
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    
    if img is None:
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.title(fila["clase_lengua"], fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
zip_final = DATA_DIR / "imgsMIIA_jaime.zip"

if zip_final.exists():
    zip_final.unlink()

shutil.make_archive(
    base_name=str(zip_final).replace(".zip", ""),
    format="zip",
    root_dir=FINAL_DIR.parent,
    base_dir=FINAL_DIR.name
)

print("ZIP final creado:")
print(zip_final)